# 018-segfix-correction

# Imports

In [ ]:
import json
import os
import time

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.multidate import load_date
from uavloc.datasets import dataset_ref
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import ProbabilityRefiner, SegmentationFeatures, load_segmenter
from uavloc.matching.benchmark import benchmark_draws, injected_priors, score_draws, summarise
from uavloc.matching.correlator import PositionCorrelator
from uavloc.segmentation.refine.segfix import SegFixProbabilities, load_segfix
from uavloc.track import REPO_ROOT, experiment_dir, log_run, set_seed

# Params

In [ ]:
EXP = "018-segfix-correction"
NOTE = ""
LOG_RUN = True

REFINER = "none"                  # "none" | "segfix"
SEGFIX_CHECKPOINT = "experiments/014-boundary-displacement/artifacts/segfix-k8-seed42.pt"
OFFSET_PX = 4
BOUNDARY_THRESHOLD = 0.5

WINDOW = "rect"
SEARCH_SCALE = 1.7
PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
MAP_DATE = "newest"
MATCHING = "weighted-ncc"
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)   # background, building, track, woody, field, water
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 10.0
HARM_MARGIN_M = 2.0
N_DIRECTIONS = 8
SEED = 42
DEVICE = "mps"
TIMING_DEVICE = "cpu"
TIMING_FRAMES = 8
TIMING_REPEATS = 3
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0
SEG_ARCHITECTURE = "lraspp"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385
FRAMES_DATASET = "village-crossdate-frames"

# Data

In [ ]:
set_seed(SEED)

In [ ]:
RUN_CONFIG = {
    "method": "segmentation", "matching": MATCHING, "class_weights": CLASS_WEIGHTS,
    "noise_m": NOISE_M, "primary_noise_m": PRIMARY_NOISE_M, "n_directions": N_DIRECTIONS,
    "window": WINDOW, "search_scale": SEARCH_SCALE, "search_radius_m": None, "prior_sigma_m": None,
    "gate": "none", "fov_w_deg": FOV_W_DEG,
    "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
    "seg_architecture": SEG_ARCHITECTURE, "seg_checkpoint_sat": SEG_CHECKPOINT, "seg_checkpoint_drone": SEG_CHECKPOINT,
    "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": MATCH_GSD_M, "match_gsd_m": MATCH_GSD_M,
    "frames_dataset": FRAMES_DATASET, "frames_dataset_ref": dataset_ref(FRAMES_DATASET),
    "places": PLACES, "map_date": MAP_DATE,
    "refiner": REFINER,
    "segfix_checkpoint": SEGFIX_CHECKPOINT if REFINER == "segfix" else None,
    "offset_px": OFFSET_PX if REFINER == "segfix" else None,
    "boundary_threshold": BOUNDARY_THRESHOLD if REFINER == "segfix" else None,
}

In [ ]:
def frame_place(frame: DroneFrame) -> str:
    return json.loads((frame.image_path.parent / "meta.json").read_text())["place"]

In [ ]:
frames = [f for f in load_frames(FRAMES_DATASET) if frame_place(f) in PLACES]
places = {place: [f for f in frames if frame_place(f) == place] for place in PLACES}
maps = {place: load_date(place, MAP_DATE) for place in PLACES}
{place: len(members) for place, members in places.items()}

# Model

In [ ]:
def make_refiner(device: str) -> ProbabilityRefiner | None:
    if REFINER == "none":
        return None
    return SegFixProbabilities(load_segfix(REPO_ROOT / SEGFIX_CHECKPOINT, device), device, OFFSET_PX, BOUNDARY_THRESHOLD)

In [ ]:
def make_correlator(place: str, device: str) -> PositionCorrelator:
    model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, device, SEG_ARCHITECTURE)
    features = SegmentationFeatures(model, model, maps[place], SEG_CHANNELS, MATCH_GSD_M, MATCH_GSD_M, device,
                                    make_refiner(device))
    return PositionCorrelator(features, config)

In [ ]:
config = correlator_config(RUN_CONFIG)
correlators = {place: make_correlator(place, DEVICE) for place in PLACES}
config

# Run

In [ ]:
draws = {place: [d for noise_m in NOISE_M
                 for d in benchmark_draws(correlators[place], places[place], noise_m, N_DIRECTIONS, SEED)]
         for place in PLACES}
{place: len(d) for place, d in draws.items()}

In [ ]:
records = pd.concat([score_draws(draws[place], config).assign(place=place) for place in PLACES], ignore_index=True)
records["harm"] = records["error_m"] > records["prior_error_m"] + HARM_MARGIN_M
records.groupby(["place", "noise_m"])[["error_m", "harm", "reachable"]].mean().round(3)

In [ ]:
subsets = {"all": {f.name for f in frames}, **{p.split("-")[0]: {f.name for f in m} for p, m in places.items()}}
metrics = summarise(records, subsets, PRIMARY_NOISE_M)
{k: round(v, 3) for k, v in metrics.items() if "_at_" not in k}

# Timing

In [ ]:
def median_ms_per_correction(correlator: PositionCorrelator, members: list[DroneFrame]) -> float:
    """Median over repeats of the mean wall time of one correction, drone inference and refiner included."""
    priors = [injected_priors(f.position, PRIMARY_NOISE_M, N_DIRECTIONS, 0.0)[0][1] for f in members]
    repeats = []
    for _ in range(TIMING_REPEATS):
        correlator._drone_maps.clear()
        start = time.perf_counter()
        for frame, prior in zip(members, priors):
            correlator.correct(frame, prior)
        repeats.append((time.perf_counter() - start) * 1000 / len(members))
    return float(np.median(repeats))

In [ ]:
timing_place = PLACES[0]
metrics["ms_per_correction"] = median_ms_per_correction(make_correlator(timing_place, TIMING_DEVICE),
                                                        places[timing_place][:TIMING_FRAMES])
metrics["ms_per_correction"]

# Plots

In [ ]:
fig_cdf, ax = plt.subplots(figsize=(6, 4))
for noise_m, group in records.groupby("noise_m"):
    errors = np.sort(group["error_m"].to_numpy())
    line, = ax.plot(errors, np.linspace(0, 1, len(errors)), label=f"{noise_m:g} m injected")
    ax.axvline(noise_m, color=line.get_color(), ls=":", lw=1)
ax.set_xlabel("position error after correction, m"); ax.set_ylabel("share of draws")
ax.set_title(f"refiner {REFINER}" + (f", s = {OFFSET_PX} px" if REFINER == "segfix" else "")); ax.legend(); ax.grid(alpha=.3)
fig_cdf.tight_layout()

In [ ]:
per_frame = records[records["noise_m"] == PRIMARY_NOISE_M].groupby("frame")["error_m"].mean().sort_values()
fig_frames, ax = plt.subplots(figsize=(12, 3.5))
per_frame.plot.bar(ax=ax, width=.8)
ax.axhline(PRIMARY_NOISE_M, color="k", ls=":", lw=1)
ax.set_ylabel(f"mean error @ {PRIMARY_NOISE_M:g} m, m"); ax.set_title("per frame, 8 directions each")
fig_frames.tight_layout()

# Log

In [ ]:
RUN_CONFIG["timing_device"] = TIMING_DEVICE
artifacts = experiment_dir(EXP) / "artifacts"
artifacts.mkdir(exist_ok=True)
run = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=[fig_cdf, fig_frames],
              seed=SEED, device=DEVICE) if LOG_RUN else {"run": "dry"}
records.to_csv(artifacts / f"records-{run['run']}.csv", index=False)
run["run"]